In [ ]:
# ==============================================================================
# CELL 0: HARDWARE VERIFICATION & DEPENDENCY SETUP
# ==============================================================================
import subprocess
import sys
import os
import torch

print('=' * 65)
print('  UNSLOTH GEMMA 4 31B LORA TRAINING SETUP (NVIDIA L4 24GB)')
print('=' * 65)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available! Ensure GPU accelerator is enabled.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)')
assert vram_gb >= 20.0, f'Expected >= 20 GB VRAM (L4/A100), found {vram_gb:.2f} GB!'

print('\nInstalling Unsloth, TRL, PEFT, and bitsandbytes with internet enabled...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'torch', 'torchvision', 'torchaudio',
    'bitsandbytes', 'transformers', 'accelerate', 'trl', 'peft'
], check=True)

try:
    import unsloth
    print('Unsloth is already installed.')
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/unslothai/unsloth.git'
    ], check=True)
    import unsloth
    print('Unsloth installed successfully.')


In [ ]:
# ==============================================================================
# CELL 1: DATASET INGESTION & ZERO-LEAK VERIFICATION
# ==============================================================================
import base64
import gzip
import json
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
DATA_DIR = WORKING_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / 'unsloth_sft_train.jsonl'
val_path = DATA_DIR / 'unsloth_sft_val.jsonl'

# Check if dataset is mounted from Kaggle Dataset input
MOUNTED_DATASET_DIR = Path('/kaggle/input/gemma4-unsloth-sft-dataset')
if (MOUNTED_DATASET_DIR / 'unsloth_sft_train.jsonl').exists():
    print(f'Loading dataset from mounted Kaggle input: {MOUNTED_DATASET_DIR}')
    train_path = MOUNTED_DATASET_DIR / 'unsloth_sft_train.jsonl'
    val_path = MOUNTED_DATASET_DIR / 'unsloth_sft_val.jsonl'
else:
    print('Mounted dataset not found; unpacking verified embedded dataset...')
    GZ_TRAIN_B64 = '...'
    GZ_VAL_B64 = '...'
    train_path.write_bytes(gzip.decompress(base64.b64decode(GZ_TRAIN_B64)))
    val_path.write_bytes(gzip.decompress(base64.b64decode(GZ_VAL_B64)))

# Verify integrity and zero host leaks
bad_markers = ['/private/var/folders', 'swegemma_sandbox', '/Users/yapilymm']
for p in [train_path, val_path]:
    content = p.read_text(encoding='utf-8')
    lines = [json.loads(line) for line in content.splitlines() if line.strip()]
    leaks = [m for m in bad_markers if m in content]
    print(f'Verified {p.name}: {len(lines)} trajectories, {len(content)} bytes, host leaks={leaks}')
    assert not leaks, f'Fatal: Leaked host markers {leaks} detected in {p.name}!'

print('Dataset ingestion verified: 100% clean and ready for training.')


In [ ]:
# ==============================================================================
# CELL 2: MODEL INITIALIZATION & RANK-8 PEFT CONFIGURATION
# ==============================================================================
import torch
from unsloth import FastLanguageModel
from pathlib import Path

max_seq_length = 16384
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
load_in_4bit = True

print(f'Loading Gemma 31B in 4-bit QLoRA (max_seq_length={max_seq_length}, dtype={dtype})...')

LOCAL_MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
FALLBACK_MODEL_PATH = Path('/kaggle/input/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')

if LOCAL_MODEL_PATH.exists():
    model_name = str(LOCAL_MODEL_PATH)
elif FALLBACK_MODEL_PATH.exists():
    model_name = str(FALLBACK_MODEL_PATH)
else:
    model_name = 'google/gemma-4-31b-it'
print(f'Using model path: {model_name}')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print('Configuring anti-memorization Rank-8 LoRA adapter on attention/output heads (q_proj, v_proj, o_proj)...')
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'v_proj', 'o_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()
print('Model initialized and PEFT LoRA configured successfully.')


In [ ]:
# ==============================================================================
# CELL 3: SFT TRAINING WITH RESPONSE-ONLY LOSS MASKING
# ==============================================================================
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

train_dataset = load_dataset('json', data_files=str(train_path), split='train')
val_dataset = load_dataset('json', data_files=str(val_path), split='train')
print(f'HuggingFace datasets loaded: train={len(train_dataset)}, val={len(val_dataset)}')

OUTPUT_DIR = WORKING_DIR / 'training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=3,
    max_steps=25,
    learning_rate=2.0e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=2,
    eval_strategy='steps',
    eval_steps=5,
    optim='adamw_8bit',
    weight_decay=0.05,
    lr_scheduler_type='cosine',
    neftune_noise_alpha=5,
    seed=42,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

# CRITICAL: Mask instruction/prompt tokens so cross-entropy loss is computed
# ONLY on model turns (<start_of_turn>model\n...<end_of_turn>).
try:
    trainer = train_on_responses_only(
        trainer,
        instruction_part='<start_of_turn>user\n',
        response_part='<start_of_turn>model\n',
    )
    print('Successfully applied response-only loss masking.')
except Exception as exc:
    print(f'Note: train_on_responses_only fallback: {exc}')

print('Starting Unsloth LoRA fine-tuning on NVIDIA L4...')
train_stats = trainer.train()
print('\nTraining complete! Train stats:')
print(train_stats)


In [ ]:
# ==============================================================================
# CELL 4: EXPORT LORA ADAPTER WEIGHTS & BUILD SUBMISSION PACKAGE
# ==============================================================================
import shutil
from pathlib import Path

ADAPTER_DIR = WORKING_DIR / 'submission' / 'adapters' / 'main_lora'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print(f'Exporting LoRA adapter weights to {ADAPTER_DIR}...')
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print('\n================ EXPORTED ADAPTER FILES ================')
total_size = 0
for p in sorted(ADAPTER_DIR.iterdir()):
    if p.is_file():
        sz = p.stat().st_size
        total_size += sz
        print(f'  - {p.name} ({sz / (1024 * 1024):.2f} MB)')
print(f'Total adapter size: {total_size / (1024 * 1024):.2f} MB')
assert (ADAPTER_DIR / 'adapter_config.json').exists(), 'Missing adapter_config.json!'
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists() or (ADAPTER_DIR / 'adapter_model.bin').exists(), 'Missing weights!'

# Verify compliance: Unpacked submission must be < 3 GiB
assert total_size < 3 * 1024 * 1024 * 1024, 'Adapter exceeds 3 GiB ADK constraint!'
print('LoRA adapter export verified and compliant with competition constraints.')
